# Four ways to mix EnergyPlus + real BAS (physics LSTM fun)

Same small **physics-residual LSTM** idea, four honest coupling modes:

| # | Mode | Train | Test / plot |
|---|---|---|---|
| 1 | Transfer | E+ IdealLoads baseline | Real BAS meter |
| 2 | Inverse | Real BAS | E+ IdealLoads |
| 3 | Joint | E+ and real (source flag) | Both held-out |
| 4 | Hybrid residual | `real − eplus` on paired days | Held-out paired day |

**Honesty:** `FUN_EXPERIMENT` / `NON_PROMOTABLE`. IdealLoads ≠ W2A plant. Never silent-merge kW columns.

Needs overlapping days in:
- `$LAKESIDE_SITE_ROOT/eplus/dsm_farm_paired/heating_dsm_eplus_paired_15min_v1.parquet`
- `$LAKESIDE_SITE_ROOT/ml/artifacts/real_baseline_15min_v1.parquet`

In [ ]:
%matplotlib inline
from datetime import datetime
print("KERNEL ALIVE", datetime.now().isoformat(timespec="seconds"), flush=True)

import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from IPython.display import display, Markdown

ROOT = Path("..").resolve()
if not (ROOT / "ml").is_dir():
    ROOT = Path(".").resolve()
sys.path[:0] = [str(ROOT), str(ROOT / "ml"), str(ROOT / "scripts")]

from notebook_plots import apply_notebook_theme, save_fig
from phys_lstm_models import PhysicsResidualLSTM, hdd65

apply_notebook_theme()
torch.manual_seed(0)
np.random.seed(0)

SITE = Path(os.environ.get("LAKESIDE_SITE_ROOT", r"C:\Users\ben\OneDrive\Desktop\testing\sp_creekside"))
FARM = SITE / "eplus" / "dsm_farm_paired" / "heating_dsm_eplus_paired_15min_v1.parquet"
REAL = SITE / "ml" / "artifacts" / "real_baseline_15min_v1.parquet"
FIG = ROOT / "reports" / "figures" / "phys_lstm_four_modes"
FIG.mkdir(parents=True, exist_ok=True)

for p in (FARM, REAL):
    if not p.is_file():
        raise FileNotFoundError(p)

print("torch", torch.__version__)
print("FARM", FARM)
print("REAL", REAL)
print("SETUP COMPLETE", flush=True)

## 0 · Load + align overlapping complete days

Keep E+ `facility_kw` and real `facility_kw` in **separate** columns forever.

In [ ]:
def complete_days(df, day_col="day", n=96):
    c = df.groupby(day_col).size()
    return set(c[c == n].index.astype(str))

farm = pd.read_parquet(FARM)
real = pd.read_parquet(REAL)
farm_b = farm[farm["strategy_id"].astype(str) == "baseline"].copy()
farm_b["day"] = farm_b["day"].astype(str)
real = real.copy()
real["day"] = real["day"].astype(str)

kw_e = "facility_kw"
kw_r = "facility_kw"
overlap = sorted(complete_days(farm_b) & complete_days(real))
if len(overlap) < 3:
    raise SystemExit(f"need ≥3 overlapping days, got {overlap}")

def pack_source(df, days, kw_col, source_name):
    rows = []
    for d in days:
        g = df[df["day"] == d].sort_values(
            "timestamp_utc" if "timestamp_utc" in df.columns else df.columns[0]
        ).reset_index(drop=True)
        if len(g) != 96:
            continue
        oat = g["oat_f"].to_numpy(np.float32)
        hdd = hdd65(oat).astype(np.float32)
        kw = g[kw_col].to_numpy(np.float32)
        zone = g["zone_temp_1F_A_f"].to_numpy(np.float32)
        occ = g["occupied"].to_numpy(np.float32) if "occupied" in g.columns else np.zeros(96, np.float32)
        t = np.arange(96, dtype=np.float32)
        zone_lag = np.roll(zone, 1); zone_lag[0] = zone[0]
        kw_lag = np.roll(kw, 1); kw_lag[0] = kw[0]
        # last feat = source flag: 0=E+, 1=real
        src = np.full(96, 1.0 if source_name == "real" else 0.0, np.float32)
        X = np.column_stack([
            oat, hdd, zone_lag, kw_lag,
            np.sin(2 * np.pi * t / 96), np.cos(2 * np.pi * t / 96),
            occ, src,
        ])
        rows.append({"day": d, "source": source_name, "X": X, "kw": kw, "zone": zone, "hdd": hdd})
    return rows

eplus_days = pack_source(farm_b, overlap, kw_e, "eplus")
real_days = pack_source(real, overlap, kw_r, "real")
by_e = {r["day"]: r for r in eplus_days}
by_r = {r["day"]: r for r in real_days}
paired = [d for d in overlap if d in by_e and d in by_r]

hold = paired[-1]  # last overlapping day held out for plots
train_days = paired[:-1]
display(Markdown(
    f"**Overlap complete days:** {len(paired)}  \n"
    f"Train: `{train_days}`  ·  Holdout plot day: **`{hold}`**"
))

# quick meter vs IdealLoads on holdout
fig, ax = plt.subplots(figsize=(11, 3.5))
h = np.arange(96) * 0.25
ax.plot(h, by_r[hold]["kw"], color="#264653", lw=2, label="real BAS facility_kw")
ax.plot(h, by_e[hold]["kw"], color="#e76f51", lw=2, label="E+ IdealLoads facility_kw")
ax.set_title(f"Same calendar day {hold} — sources disagree (expected)")
ax.set_xlabel("Hour ending"); ax.set_ylabel("kW"); ax.legend(frameon=False)
fig.tight_layout(); save_fig(FIG / "holdout_real_vs_eplus_raw.png", fig); plt.show()

In [ ]:
def fit_phys(hdd, kw):
    A = np.column_stack([np.ones_like(hdd), hdd])
    coef, *_ = np.linalg.lstsq(A, kw, rcond=None)
    return float(coef[0]), float(coef[1])

def phys(a, b, hdd):
    return a + b * hdd

def train_lstm(X_list, y_kw_list, hdd_list, *, epochs=60, hidden=48, lr=3e-3):
    """Train PhysicsResidualLSTM on residual kw (and dummy zone residual)."""
    X = np.stack(X_list).astype(np.float32)
    H = np.stack(hdd_list).astype(np.float32)
    K = np.stack(y_kw_list).astype(np.float32)
    a, b = fit_phys(H.ravel(), K.ravel())
    Y = np.stack([
        np.column_stack([K[i] - phys(a, b, H[i]), np.zeros(96, np.float32)])
        for i in range(len(X))
    ]).astype(np.float32)
    model = PhysicsResidualLSTM(n_in=X.shape[-1], hidden=hidden, n_out=2)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    xt, yt = torch.tensor(X), torch.tensor(Y)
    losses = []
    model.train()
    for _ in range(epochs):
        opt.zero_grad()
        pred = model(xt)
        loss = nn.functional.smooth_l1_loss(pred[..., 0], yt[..., 0])
        loss.backward(); opt.step()
        losses.append(float(loss.detach()))
    model.eval()
    return model, a, b, losses

def predict_kw(model, a, b, rec):
    with torch.no_grad():
        res = model(torch.tensor(rec["X"][None]))[0, :, 0].numpy()
    return phys(a, b, rec["hdd"]) + res

def mae(y, yhat):
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(yhat))))

score_rows = []
hours = np.arange(96) * 0.25

## Mode 1 — Train on E+, test on real BAS

Question: does an IdealLoads-trained residual LSTM transfer to the meter?

In [ ]:
m1, a1, b1, loss1 = train_lstm(
    [by_e[d]["X"] for d in train_days],
    [by_e[d]["kw"] for d in train_days],
    [by_e[d]["hdd"] for d in train_days],
)
# evaluate on real holdout (force source flag=1 in features for honesty of joint layout;
# model was trained with flag=0 — transfer stress test)
rec_r = by_r[hold]
yhat1 = predict_kw(m1, a1, b1, rec_r)
m_phys1 = mae(rec_r["kw"], phys(a1, b1, rec_r["hdd"]))
m_lstm1 = mae(rec_r["kw"], yhat1)
score_rows.append({"mode": "1_train_eplus_test_real", "mae_physics": m_phys1, "mae_model": m_lstm1})

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(hours, rec_r["kw"], color="#264653", lw=2.2, label="real BAS")
ax.plot(hours, phys(a1, b1, rec_r["hdd"]), "--", color="#2a9d8f", label="HDD prior (fit on E+)")
ax.plot(hours, yhat1, color="#e76f51", lw=2, label="E+-trained LSTM → real")
ax.set_title(f"Mode 1 · {hold} · MAE phys={m_phys1:.1f} · LSTM={m_lstm1:.1f} kW")
ax.set_xlabel("Hour ending"); ax.set_ylabel("kW"); ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); save_fig(FIG / "mode1_transfer.png", fig); plt.show()

## Mode 2 — Train on real BAS, plot vs E+

Question: does a meter-trained model look anything like IdealLoads that day?

In [ ]:
m2, a2, b2, loss2 = train_lstm(
    [by_r[d]["X"] for d in train_days],
    [by_r[d]["kw"] for d in train_days],
    [by_r[d]["hdd"] for d in train_days],
)
rec_e = by_e[hold]
yhat2 = predict_kw(m2, a2, b2, rec_e)
m_phys2 = mae(rec_e["kw"], phys(a2, b2, rec_e["hdd"]))
m_lstm2 = mae(rec_e["kw"], yhat2)
score_rows.append({"mode": "2_train_real_test_eplus", "mae_physics": m_phys2, "mae_model": m_lstm2})

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(hours, rec_e["kw"], color="#e76f51", lw=2.2, label="E+ IdealLoads")
ax.plot(hours, by_r[hold]["kw"], color="#264653", alpha=0.45, lw=1.5, label="real BAS (same day)")
ax.plot(hours, yhat2, color="#2a9d8f", lw=2, label="real-trained LSTM → E+")
ax.set_title(f"Mode 2 · {hold} · MAE phys={m_phys2:.1f} · LSTM={m_lstm2:.1f} kW")
ax.set_xlabel("Hour ending"); ax.set_ylabel("kW"); ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); save_fig(FIG / "mode2_inverse.png", fig); plt.show()

## Mode 3 — Joint train (source flag in features)

Stack E+ and real sequences; feature dim 7 is `source` (0=E+, 1=real).  
Held-out: predict real and E+ for `hold` separately.

In [ ]:
Xj, Kj, Hj = [], [], []
for d in train_days:
    Xj += [by_e[d]["X"], by_r[d]["X"]]
    Kj += [by_e[d]["kw"], by_r[d]["kw"]]
    Hj += [by_e[d]["hdd"], by_r[d]["hdd"]]

m3, a3, b3, loss3 = train_lstm(Xj, Kj, Hj, epochs=80)
yhat3_r = predict_kw(m3, a3, b3, by_r[hold])
yhat3_e = predict_kw(m3, a3, b3, by_e[hold])
m3r = mae(by_r[hold]["kw"], yhat3_r)
m3e = mae(by_e[hold]["kw"], yhat3_e)
score_rows.append({"mode": "3_joint_on_real", "mae_physics": mae(by_r[hold]["kw"], phys(a3, b3, by_r[hold]["hdd"])), "mae_model": m3r})
score_rows.append({"mode": "3_joint_on_eplus", "mae_physics": mae(by_e[hold]["kw"], phys(a3, b3, by_e[hold]["hdd"])), "mae_model": m3e})

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(hours, by_r[hold]["kw"], color="#264653", lw=2, label="real")
axes[0].plot(hours, yhat3_r, color="#2a9d8f", lw=2, label="joint LSTM (source=real)")
axes[0].set_ylabel("kW"); axes[0].legend(frameon=False); axes[0].set_title(f"Mode 3a · real · MAE={m3r:.1f}")
axes[1].plot(hours, by_e[hold]["kw"], color="#e76f51", lw=2, label="E+")
axes[1].plot(hours, yhat3_e, color="#457b9d", lw=2, label="joint LSTM (source=eplus)")
axes[1].set_xlabel("Hour ending"); axes[1].set_ylabel("kW")
axes[1].legend(frameon=False); axes[1].set_title(f"Mode 3b · E+ · MAE={m3e:.1f}")
fig.tight_layout(); save_fig(FIG / "mode3_joint.png", fig); plt.show()

## Mode 4 — Hybrid residual (closest to product idea)

On paired days: target = `real_kw − eplus_kw`.  
Features use **real** weather/lags + source flag=1.  
Reconstruction: `ŷ_real = eplus_kw + LSTM(Δ)`.

In [ ]:
# residual target series per train day
X4, dK, H4 = [], [], []
for d in train_days:
    X4.append(by_r[d]["X"])  # real-side features
    dK.append(by_r[d]["kw"] - by_e[d]["kw"])
    H4.append(by_r[d]["hdd"])

# For residual mode, skip HDD prior on Δ (fit near-zero prior)
X = np.stack(X4).astype(np.float32)
Ydelta = np.stack(dK).astype(np.float32)
Y = np.stack([np.column_stack([Ydelta[i], np.zeros(96, np.float32)]) for i in range(len(X))]).astype(np.float32)
m4 = PhysicsResidualLSTM(n_in=8, hidden=48, n_out=2)
opt = torch.optim.Adam(m4.parameters(), lr=3e-3)
xt, yt = torch.tensor(X), torch.tensor(Y)
m4.train()
for _ in range(80):
    opt.zero_grad()
    pred = m4(xt)
    loss = nn.functional.smooth_l1_loss(pred[..., 0], yt[..., 0])
    loss.backward(); opt.step()
m4.eval()
with torch.no_grad():
    dhat = m4(torch.tensor(by_r[hold]["X"][None]))[0, :, 0].numpy()
yhat4 = by_e[hold]["kw"] + dhat
m_e_as_real = mae(by_r[hold]["kw"], by_e[hold]["kw"])
m4m = mae(by_r[hold]["kw"], yhat4)
score_rows.append({"mode": "4_hybrid_residual", "mae_physics": m_e_as_real, "mae_model": m4m})

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(hours, by_r[hold]["kw"], color="#264653", lw=2.2, label="real BAS")
axes[0].plot(hours, by_e[hold]["kw"], color="#e76f51", ls="--", lw=1.6, label="E+ IdealLoads")
axes[0].plot(hours, yhat4, color="#2a9d8f", lw=2, label="E+ + LSTM(Δ)")
axes[0].set_ylabel("kW"); axes[0].legend(frameon=False, fontsize=8)
axes[0].set_title(f"Mode 4 · {hold} · MAE(E+ as real)={m_e_as_real:.1f} · hybrid={m4m:.1f} kW")
axes[1].plot(hours, by_r[hold]["kw"] - by_e[hold]["kw"], color="#264653", label="true Δ (real−E+)")
axes[1].plot(hours, dhat, color="#2a9d8f", label="LSTM Δ")
axes[1].axhline(0, color="#999", lw=0.8)
axes[1].set_xlabel("Hour ending"); axes[1].set_ylabel("Δ kW")
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout(); save_fig(FIG / "mode4_hybrid_residual.png", fig); plt.show()

## Scoreboard

In [ ]:
score = pd.DataFrame(score_rows)
score["lift_vs_physics"] = score["mae_physics"] - score["mae_model"]
display(score)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(score))
w = 0.35
ax.bar(x - w/2, score["mae_physics"], w, label="baseline (physics or raw E+)", color="#8d99ae")
ax.bar(x + w/2, score["mae_model"], w, label="model", color="#2a9d8f")
ax.set_xticks(x)
ax.set_xticklabels(score["mode"], rotation=20, ha="right", fontsize=8)
ax.set_ylabel("MAE kW on holdout")
ax.set_title("Four coupling modes — lower model bar is better")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); save_fig(FIG / "four_modes_scoreboard.png", fig); plt.show()

card = {
    "honesty": "FUN_EXPERIMENT",
    "promote": "NON_PROMOTABLE",
    "holdout_day": hold,
    "train_days": train_days,
    "overlap_days": paired,
    "score": score.to_dict(orient="records"),
    "note": (
        "Compares E+/BAS coupling modes with a tiny physics-residual LSTM. "
        "IdealLoads farm is STRUCTURAL_LOAD_DIAGNOSTIC — not W2A plant truth."
    ),
}
(FIG / "four_modes_card.json").write_text(json.dumps(card, indent=2), encoding="utf-8")
display(Markdown(
    "**How to read:** Mode 4 is closest to the hybrid product story. "
    "Mode 1 usually looks bad (domain shift). Mode 3 only helps if the source flag is respected. "
    "Still **non-promotable** fun."
))
print("wrote", FIG / "four_modes_card.json")